In [ ]:
import os
import re
import json
import time
import csv
import requests
import sys
from pathlib import Path
from google.colab import files
from datetime import datetime


class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "a", encoding="utf-8")

    def write(self, message):
        # This writes to the console
        self.terminal.write(message)
        # This writes to the file
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()

    def close(self):
        self.log.close()


def read_env_value(env_path, key_name):
    env_path = Path(env_path)
    if not env_path.exists():
        return None

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if "=" not in line:
            continue

        key, value = line.split("=", 1)
        if key.strip() != key_name:
            continue

        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in ('"', "'"):
            value = value[1:-1]
        return value

    return None

# ERROR CODES:

**FINAL_NUMERIC_IS_NOT_NUMBER** = final_numeric is not a number

**FINAL_NUMERIC_IS_NULL** = final_numeric is null

**FINAL_FORMULA_IS_NOT_STRING** = final_formula is not string

**FINAL_UNITS_IS_NOT_STRING** = final_units is not string

**MISSING_OUTPUT_FIELD** = missing output field


In [ ]:
PROBLEM_MAX = 1
PROBLEM_START = 14
PROMPT_VERSION = 0 # first version, taking from PROMPT_TEMPLATES[] (below)

INPUT_CSV = "/content/with examples Physics Problems - with examples - Physics Problems (final).csv"

OUTPUT_FOLDER = "/content/"

OUTPUT_FILENAME = "physics_experiment"

AI_MODELS = [
  #"anthropic/claude-opus-4.7"
  #"anthropic/claude-sonnet-4.5"
  #"deepseek/deepseek-r1"
  #"deepseek/deepseek-v3.2-speciale"
  #"google/gemini-2.5-pro"
  #"google/gemini-3.1-pro-preview"
  #"google/gemma-4-31b-it"
  #"meta-llama/llama-4-maverick"
  #"microsoft/phi-4"
  #"mistralai/ministral-14b-2512"
  #"mistralai/pixtral-large-2411"
  #"moonshotai/kimi-k2.5"
  #"moonshotai/kimi-k2.6"
  #"nvidia/llama-3.3-nemotron-super-49b-v1.5"
  #"openai/gpt-5.4-pro"
  #"openai/gpt-5.4"
  #"openai/gpt-oss-120b"
  #"openai/o4-mini-high"
  #"qwen/qwen3.5-397b-a17b"
  #"qwen/qwen3.6-plus"
  "stepfun/step-3.5-flash"
  #"x-ai/grok-4.20"
  #"xiaomi/mimo-v2.5-pro"
]

###########

OPENROUTER_API_KEY =  ''


AI_MODEL_DEFAULT_TEMPERATURE = 0.1
API_URL = "https://openrouter.ai/api/v1/chat/completions"

REQUEST_TIMEOUT = 1800
MAX_RETRIES = 2
SLEEP_BETWEEN_REQUESTS_SEC = 1.0


REQUIRED_OUTPUT_FIELDS = ["final_numeric", "final_formula", "final_units"]

if not OPENROUTER_API_KEY:
    raise ValueError(
        f"Could not load OpenRouter API key. Expected OPENROUTER_API_KEY"
    )


**PROMPT**

In [ ]:
PROMPT_TEMPLATES = [
# version 1 (index 0)
"""
You are an expert in physics problems.

# INPUT

The following is a physics problem that needs to be solved.

{{PROBLEM}}

# OUTPUT REQUIREMENTS

Answer must be only in JSON format. Do not use markdown. Do not add any extra text.

Return a single JSON object with these required keys:
- "final_numeric": the final numeric value as a JSON number (not a string). If the answer is symbolic-only, use null.
- "final_formula": a string containing the final answer as a formula/expression written as valid LaTeX.
- "final_units": a string with units in International System of Units (SI) that was used for the numeric answer.
If answer is unitless, use "1".
If the numeric answer is null, still provide the expected units for the expression.
If the problem does not explicitly specify units, default to SI units.

IMPORTANT: answer only with JSON, do not add any comments or other text execpt output JSON with results.

# OUTPUT FORMAT - EXAMPLE 1:

{
  "final_numeric": 1,
  "final_formula": "LATEX_FORMULA_INSERT_HERE",
  "final_units": "N" // NEWTON units
}

# OUTPUT FORMAT - EXAMPLE 2:

{
  "final_numeric": 1,
  "final_formula": "LATEX_FORMULA_INSERT_HERE",
  "final_units": "J" // JOULE units
}

# OUTPUT FORMAT - EXAMPLE 3:

{
  "final_numeric": 1,
  "final_formula": "LATEX_FORMULA_INSERT_HERE",
  "final_units": "kg*m/s^2" // kilogram multiplied by meters per second squared
}

# INCORRECT OUTPUT FORMAT EXAMPLE 1:

{
  "final_numeric": "zero",
  "final_formula": "m  / divide by n",
  "final_units": "KILOGRAMS"
}

# INCORRECT OUTPUT FORMAT EXAMPLE 1:

{
  "final_numeric": [11,0],
  "final_formula": "m  / divide by n",
  "final_units": "KILOGRAMS"
}

""",
  # version 2 goes here
  """
  """
]

**SPREADSHEET**

In [ ]:
# TODO (EM): Change to CSV file as input


#uploaded = files.upload()

#xlsx_files = [name for name in uploaded.keys() if name.lower().endswith(".xlsx")]
#if not xlsx_files:
#    raise ValueError("Please upload your .xlsx spreadsheet file.")
#
#INPUT_XLSX = OUTPUT_FOLDER + xlsx_files[0]
#print("Using file:", INPUT_XLSX)




In [ ]:
def normalize_header(value):
    return str(value).strip().lower() if value is not None else ""

def load_problems_from_file(csv_path, start_problem=1, max_items=25):
    items = []

    print(f"Loading problems from {csv_path}, start_problem={start_problem}, max_items={max_items}")

    with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)

        if reader.fieldnames is None:
            raise ValueError("CSV file is empty or missing a header row.")

        statement_field = None
        for field in reader.fieldnames:
            if normalize_header(field) == "statement":
                statement_field = field
                break

        if statement_field is None:
            raise KeyError(f'Column "Statement" not found. Headers seen: {reader.fieldnames}')

        # i is the index of row in CSV ()
        for i, row in enumerate(reader):
            # check if have statement is not empty
            statement = row.get(statement_field)

            if statement is None:
                continue

            # also skip if not reached start problem yet
            if i < start_problem-1:
                continue

            statement = str(statement).strip()
            if not statement:
                continue

            items.append({
                # we add +1 because have first row is HEADER (1 row)
                "statement_row_id": i+1,
                "statement": statement
            })

            if len(items) >= max_items:
                break

    if not items:
        raise ValueError('No non-empty rows found from the specified start item in the "Statement" column.')

    return items

In [ ]:
def build_user_prompt(problem_text):
    return PROMPT_TEMPLATES[PROMPT_VERSION].replace("{{PROBLEM}}", problem_text)

def extract_message_text(content):
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                if item.get("type") == "text":
                    parts.append(item.get("text", ""))
                elif "text" in item:
                    parts.append(str(item["text"]))
        return "\n".join(p for p in parts if p).strip()
    return str(content).strip()

PARSING_ERROR_CODE = "PARSING_ERROR"

def _append_parse_error(obj, code=PARSING_ERROR_CODE):
    if not isinstance(obj, dict):
        return obj

    existing = str(obj.get("error", "") or "").strip()
    parts = [p.strip() for p in existing.splitlines() if p.strip()]

    if code not in parts:
        parts.append(code)

    obj["error"] = "\n".join(parts)
    return obj

def _strip_json_code_fence(text):
    text = str(text).strip()

    fenced = re.match(
        r"^```(?:json)?\s*(.*?)\s*```$",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    if fenced:
        return fenced.group(1).strip()

    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def _escape_invalid_json_backslashes(text):
    r"""
    Repair common LLM JSON mistakes in LaTeX strings.

    Valid JSON escapes are preserved:
      \", \\, \/, \b, \f, \n, \r, \t, \u1234

    Invalid LaTeX-style escapes are repaired:
      \frac  -> \\frac
      \text  -> \\text
      \rho   -> \\rho
      \_     -> \\_
    """
    out = []
    i = 0
    hex_chars = set("0123456789abcdefABCDEF")

    while i < len(text):
        ch = text[i]

        if ch != "\\":
            out.append(ch)
            i += 1
            continue

        if i + 1 >= len(text):
            out.append("\\\\")
            i += 1
            continue

        nxt = text[i + 1]

        if nxt in ['"', "\\", "/", "b", "f", "n", "r", "t"]:
            out.append("\\" + nxt)
            i += 2
        elif (
            nxt == "u"
            and i + 5 < len(text)
            and all(c in hex_chars for c in text[i + 2:i + 6])
        ):
            out.append(text[i:i + 6])
            i += 6
        else:
            out.append("\\\\")
            i += 1

    return "".join(out)

def _raw_decode_json_prefix(text):
    r"""
    Parse a JSON object from the beginning of the response.

    Unlike json.loads(), JSONDecoder.raw_decode() can parse this case:

        {"final_numeric": 1, ...}
        Wait, let me recalculate...

    The object is marked with PARSING_ERROR because the full response was
    not valid JSON-only output.
    """
    decoder = json.JSONDecoder()

    candidates = [text]
    repaired = _escape_invalid_json_backslashes(text)
    if repaired != text:
        candidates.append(repaired)

    for candidate in candidates:
        try:
            obj, end = decoder.raw_decode(candidate)
            if isinstance(obj, dict):
                if candidate[end:].strip():
                    _append_parse_error(obj)
                if candidate != text:
                    _append_parse_error(obj)
                return obj
        except json.JSONDecodeError:
            pass

    return None

# we use this function cleanup JSON input string like this:
#{
#  "final_numeric": 0.32,
#  "final_formula": "\\frac{d \cdot v_{truck}}{v_{truck}^2 + v_{car}^2}",
#  "final_units": "h"
#}
# it failed to be parsed as JSON because of LaTeX formular having \frac instead of \\frac

def _extract_first_balanced_json_object(text):
    r"""
    Return the first balanced {...} block without using a greedy regex.

    This intentionally ignores braces inside JSON strings, so LaTeX like
    "\\frac{a}{b}" does not break object-boundary detection.
    """
    for start in [m.start() for m in re.finditer(r"\{", text)]:
        depth = 0
        in_string = False
        escaped = False

        for i in range(start, len(text)):
            ch = text[i]

            if in_string:
                if escaped:
                    escaped = False
                elif ch == "\\":
                    escaped = True
                elif ch == '"':
                    in_string = False
            else:
                if ch == '"':
                    in_string = True
                elif ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        return text[start:i + 1]

    return None

def _loads_json_object_candidate(candidate):
    for fixed in (candidate, _escape_invalid_json_backslashes(candidate)):
        try:
            obj = json.loads(fixed)
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            pass

    return None

def _extract_quoted_value(text, start_quote_index):
    chars = []
    escaped = False
    i = start_quote_index + 1

    while i < len(text):
        ch = text[i]

        if escaped:
            chars.append(ch)
            escaped = False
        elif ch == "\\":
            chars.append(ch)
            escaped = True
        elif ch == '"':
            return "".join(chars), i + 1
        else:
            chars.append(ch)

        i += 1

    # Response was truncated before the closing quote.
    return "".join(chars), len(text)

def _extract_bracketed_value(text, start_index):
    opening = text[start_index]
    closing = "]" if opening == "[" else "}"

    depth = 0
    in_string = False
    escaped = False

    for i in range(start_index, len(text)):
        ch = text[i]

        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == opening:
                depth += 1
            elif ch == closing:
                depth -= 1
                if depth == 0:
                    raw = text[start_index:i + 1]
                    try:
                        return json.loads(_escape_invalid_json_backslashes(raw)), i + 1
                    except Exception:
                        return raw, i + 1

    # Response was truncated before the bracket closed.
    return text[start_index:], len(text)

def _extract_value_after_key(text, key):
    match = re.search(r'"' + re.escape(key) + r'"\s*:', text)
    if not match:
        return None, False

    i = match.end()

    while i < len(text) and text[i].isspace():
        i += 1

    if i >= len(text):
        return None, False

    if text[i] == '"':
        value, _ = _extract_quoted_value(text, i)
        return value, True

    if text[i] in "[{":
        value, _ = _extract_bracketed_value(text, i)
        return value, True

    rest = text[i:]
    scalar_match = re.match(
        r"(null|true|false|-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)",
        rest
    )
    if scalar_match:
        raw = scalar_match.group(1)
        try:
            return json.loads(raw), True
        except Exception:
            return raw, True

    raw = re.split(r",|\n|\}", rest, maxsplit=1)[0].strip()
    return raw, bool(raw)

def _salvage_required_fields(text):
    """
    Last-resort partial extraction for malformed/truncated objects.

    This is intentionally conservative: it only extracts the required output
    keys and then lets check_fields() validate missing/wrongly typed fields.
    """
    obj = {}

    for key in REQUIRED_OUTPUT_FIELDS:
        value, found = _extract_value_after_key(text, key)
        if found:
            obj[key] = value

    if obj:
        return _append_parse_error(obj)

    return None

def find_and_extract_json_object_from_text(text):
    text = _strip_json_code_fence(text)

    # 1) Strict parse. Clean JSON-only responses stay clean.
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except json.JSONDecodeError:
        pass

    # 2) Parse JSON at the start, even if the model added explanation after it.
    obj = _raw_decode_json_prefix(text)
    if obj is not None:
        return obj

    # 3) Extract the first balanced JSON object without greedy "{.*}" matching.
    candidate = _extract_first_balanced_json_object(text)
    if candidate:
        obj = _loads_json_object_candidate(candidate)
        if obj is not None:
            return _append_parse_error(obj)

    # 4) Last resort: salvage required fields from malformed/truncated JSON.
    return _salvage_required_fields(text)


def check_fields(obj):
    if obj is None:
      print("Obj is None?")
      return False

    if "error" not in obj:
      obj["error"] = ""

    error_message = obj["error"]

    for key in REQUIRED_OUTPUT_FIELDS:
        if key not in obj:
            error_message = error_message + "\n" + "MISSING_OUTPUT_FIELD:" + key
            print(f"MISSING_OUTPUT_FIELD Missing required key: {key}")

    num = obj.get("final_numeric")
    if isinstance(num, str):
        stripped = num.strip()
        if stripped.lower() == "null":
            obj["final_numeric"] = None
        else:
            try:
                n = float(stripped)
                if n.is_integer(): n = int(n)
                obj["final_numeric"] = n
            except:
                error_message += "\nFINAL_NUMERIC_IS_NOT_NUMBER"
    elif num is not None and not isinstance(num, (int, float)):
        error_message += "\nFINAL_NUMERIC_IS_NOT_NUMBER"
        obj["final_numeric"] = str(num)

    if not isinstance(obj.get("final_formula"), str):
        error_message += "\nFINAL_FORMULA_IS_NOT_STRING"
        obj["final_formula"] = str(obj.get("final_formula", ""))

    if not isinstance(obj.get("final_units"), str):
        error_message += "\nFINAL_UNITS_IS_NOT_STRING"
        obj["final_units"] = str(obj.get("final_units", ""))

    obj["error"] = error_message.strip()
    return True

def call_ai(ai_model, problem_id, problem_text, api_key):
    if not api_key or api_key == "PASTE_YOUR_OPENROUTER_API_KEY_HERE":
        raise ValueError("Please set OPENROUTER_API_KEY first.")

    payload = {
        "model": ai_model,
        "temperature": AI_MODEL_DEFAULT_TEMPERATURE,
        "messages": [{"role": "user", "content": build_user_prompt(problem_text)}]
    }
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:

            print(f"sending attempt: {attempt} of {MAX_RETRIES+1}")

            # sending request to AI
#            response = requests.post(API_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
#            response.raise_for_status()
            # got data back
#            data = response.json()

            response = requests.post(API_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)

            if response.status_code >= 400:
                print("STATUS:", response.status_code)
                print("BODY:", response.text)
                print("PAYLOAD:", json.dumps(payload, ensure_ascii=False, indent=2))
                response.raise_for_status()

            data = response.json()



            raw_text = extract_message_text(data["choices"][0]["message"]["content"])
            print(f"problem id: {problem_id}, raw response extracted.")

            parsed = find_and_extract_json_object_from_text(raw_text)

            if parsed is None:
                print(f"JSON can not be parsed: {raw_text}")
                return {"ok": False, "parsed": None, "raw_response_text": raw_text, "error": "JSON_PARSE_ERROR"}

            is_valid = check_fields(parsed)

            return {
                "ok": is_valid,
                "parsed": parsed,
                "raw_response_text": raw_text,
                "full_api_response": data,
                "prompt_tokens": data["usage"]["prompt_tokens"],
                "completion_tokens": data["usage"]["completion_tokens"],
                "total_tokens": data["usage"]["total_tokens"],
                "prompt_version": PROMPT_VERSION

            }

        except Exception as e:
            last_error = e
            time.sleep(min(2 ** attempt, 20))

    return {"ok": False, "parsed": None, "raw_response_text": None, "error": str(last_error)}

**Save outputs**

In [ ]:
def save_results_csv_and_jsonl(results, csv_path, jsonl_path):

    # saving to JSONL
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for row in results:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    fieldnames = [
        "datetime",
        "ai_model",
        "statement_row_id",
        "statement",
        "ok",
        "final_numeric",
        "final_formula",
        "final_units",
        "error",
        "raw_response_text",
        "full_api_response",
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
        "prompt_version"
    ]

    # saving to CSV
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in results:
            writer.writerow({
                "datetime":          row.get("datetime"),
                "ai_model":          row.get("ai_model"),
                "statement_row_id":  row.get("statement_row_id"),
                "statement":         row.get("statement"),
                "ok":                row.get("ok"),
                "final_numeric":     row.get("final_numeric"),
                "final_formula":     row.get("final_formula"),
                "final_units":       row.get("final_units"),
                "error":             row.get("error"),
                "raw_response_text": row.get("raw_response_text"),
                "full_api_response": row.get("full_api_response"),
                "prompt_tokens":     row.get("prompt_tokens"),
                "completion_tokens": row.get("completion_tokens"),
                "total_tokens":      row.get("total_tokens"),
                "prompt_version":    row.get("prompt_version")

            })

In [ ]:

# Setup Logging
now = datetime.now()
datetime_string = now.strftime("%Y-%m-%d-%H-%M-%S")

# subfolder name
current_subfolder = Path(OUTPUT_FOLDER) / datetime_string

# Create the directory structure
# parents=True: creates all intermediate folders (e.g., 'parent_folder' and 'subfolder')
# exist_ok=True: prevents an error if the folders already exist
current_subfolder.mkdir(parents=True, exist_ok=True)

log_file =   current_subfolder / f"{datetime_string}_{OUTPUT_FILENAME}.log"
file_csv =   current_subfolder / f"{datetime_string}_{OUTPUT_FILENAME}.csv"
file_jsonl = current_subfolder / f"{datetime_string}_{OUTPUT_FILENAME}.jsonl"

sys.stdout = Logger(log_file)

try:
    problems = load_problems_from_file(INPUT_CSV, start_problem=PROBLEM_START, max_items=PROBLEM_MAX)
    print(f"Loaded {len(problems)} problems from the Statement column.")
    results = []

    for idx, item in enumerate(problems, start=1):
        statement_row_id = item["statement_row_id"]
        statement = item["statement"]

        print(f"[{idx}/{len(problems)}] Sending problem #{statement_row_id} (csv row {statement_row_id+1})..")

        for ai_model in AI_MODELS:
          api_result = call_ai(
              ai_model,  # ai model
              statement_row_id, # problem id from CSV file (first column)
              statement,  # problem statement
              OPENROUTER_API_KEY
              )
          now = datetime.now()
          formatted_datetime = now.strftime("%Y-%m-%d-%H-%M-%S")

          if api_result["ok"]:
              parsed = api_result["parsed"]
              out = {
                  "ok": True,
                  "datetime": formatted_datetime,
                  "ai_model": ai_model,
                  "statement_row_id": statement_row_id,
                  "statement": statement,
                  "final_numeric": parsed.get("final_numeric"),
                  "final_formula": parsed.get("final_formula"),
                  "final_units": parsed.get("final_units"),
                  "error": parsed["error"],
                  "raw_response_text": api_result.get("raw_response_text"),
                  "full_api_response": api_result.get("full_api_response"),
                  "prompt_tokens": api_result.get("prompt_tokens"),
                  "completion_tokens": api_result.get("completion_tokens"),
                  "total_tokens": api_result.get("total_tokens"),
                  "prompt_version": PROMPT_VERSION

              }
              print("  OK:", out)
          else:
              out = {
                  "ok": False,
                  "datetime": formatted_datetime,
                  "ai_model": ai_model,
                  "statement_row_id": statement_row_id,
                  "statement": statement,
                  "final_numeric": None,
                  "final_formula": None,
                  "final_units": None,
                  "error": api_result.get("error"),
                  "raw_response_text": api_result.get("raw_response_text"),
                  "full_api_response": api_result.get("full_api_response"),
                  "prompt_tokens": api_result.get("prompt_tokens"),
                  "completion_tokens": api_result.get("completion_tokens"),
                  "total_tokens": api_result.get("total_tokens"),
                  "prompt_version": PROMPT_VERSION
              }
              print("  FAILED:", out["error"])

          results.append(out)
          time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # save csv and jsonl to files
    save_results_csv_and_jsonl(results,file_csv,file_jsonl)


finally:
    # Always restore original stdout and download log
    logger_inst = sys.stdout
    sys.stdout = logger_inst.terminal
    logger_inst.close()
    files.download(log_file)
    print("LOG file saved to: ", log_file)

    if 'file_csv' in locals():
        files.download(file_csv)
        files.download(file_jsonl)
        print("CSV saved to:", file_csv)
        print("JSONL saved to:", file_jsonl)


    print("\nDone.")

Loading problems from /content/with examples Physics Problems - with examples - Physics Problems (final).csv, start_problem=14, max_items=1
Loaded 1 problems from the Statement column.
[1/1] Sending problem #14 (csv row 15)..
sending attempt: 1 of 3
sending attempt: 2 of 3
problem id: 14, raw response extracted.
  OK: {'ok': True, 'datetime': '2026-07-22-16-05-20', 'ai_model': 'stepfun/step-3.5-flash', 'statement_row_id': 14, 'statement': 'На дне сосуда, заполненного водой, лежит плоское зеркало. Человек, наклонившийся над сосудом, видит изображение своего лица в зеркале на расстоянии d = 25 см, если расстояние от лица до поверхности воды һ = 5 см. Определить глубину сосуда l в см.\n\nExamples:\n\nExample 1: \n\n*Calculate the power of the eye when viewing an object 3.00 m away.*\n\nSolution\u2003Using the lens-to-retina distance of 2.00 cm and the equation (P = \\frac{1}{d_o} + \\frac{1}{d_i}) we can\ndetermine the power at an object distance of 3.00 m:\n\n[\nP = \\frac{1}{d_o} + \\fr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

LOG file saved to:  /content/2026-07-22-15-38-30/2026-07-22-15-38-30_physics_experiment.log


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CSV saved to: /content/2026-07-22-15-38-30/2026-07-22-15-38-30_physics_experiment.csv
JSONL saved to: /content/2026-07-22-15-38-30/2026-07-22-15-38-30_physics_experiment.jsonl

Done.
